In [2]:
%pwd


'c:\\Users\\shran\\OneDrive\\Desktop\\medical-chatbot\\medical-chatbot\\research'

In [3]:
import os 
os.chdir("../")
%pwd


'c:\\Users\\shran\\OneDrive\\Desktop\\medical-chatbot\\medical-chatbot'

In [6]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
#extract text from pdf file (path as input)
def load_pdf_files(data):
    loader = DirectoryLoader(
        data, 
        glob="*.pdf", 
        show_progress=True, 
        loader_cls=PyPDFLoader
        )
    documents = loader.load()
    return documents

In [8]:
extracted_data = load_pdf_files("data")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:27<00:00, 27.26s/it]


In [11]:
len(extracted_data)
#extracted_data

637

In [12]:
#filter operation
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    given a list of documnent objects, return a new list of document objects with 
    only 'SOURCE' in metadata and the original page content 
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src= doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content, 
                metadata={"source": src}
            )
        )
    return minimal_docs


In [15]:
minimal_docs = filter_to_minimal_docs(extracted_data)
minimal_docs[10]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='Rhonda Cloos, R.N.\nMedical Writer\nAustin, TX\nGloria Cooksey, C.N.E\nMedical Writer\nSacramento, CA\nAmy Cooper, M.A., M.S.I.\nMedical Writer\nVermillion, SD\nDavid A. Cramer, M.D.\nMedical Writer\nChicago, IL\nEsther Csapo Rastega, R.N., B.S.N.\nMedical Writer\nHolbrook, MA\nArnold Cua, M.D.\nPhysician\nBrooklyn, NY\nTish Davidson, A.M.\nMedical Writer\nFremont, California\nDominic De Bellis, Ph.D.\nMedical Writer/Editor\nMahopac, NY\nLori De Milto\nMedical Writer\nSicklerville, NJ\nRobert S. Dinsmoor\nMedical Writer\nSouth Hamilton, MA\nStephanie Dionne, B.S.\nMedical Writer\nAnn Arbor, MI\nMartin W. Dodge, Ph.D.\nTechnical Writer/Editor\nCentinela Hospital and Medical\nCenter\nInglewood, CA\nDavid Doermann\nMedical Writer\nSalt Lake City, UT\nStefanie B. N. Dugan, M.S.\nGenetic Counselor\nMilwaukee, WI\nDoug Dupler, M.A.\nScience Writer\nBoulder, CO\nJulie A. Gelderloos\nBiomedical Writer\nPlaya del Rey, CA\nGar

In [ ]:
#chunking operation
#split the documents into smaller chunks (for better processing)
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, #500 token one chunk
        chunk_overlap=20, #for understanding the context, we need to have some overlap between chunks
        )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [17]:
texts_chunk = text_split(minimal_docs)
print(len(texts_chunk))

5859


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
